# 2 · An address book: relations instead of arrays

**The problem.** A contact has several addresses, each with a label ("home", "work"). The obvious model is an array
of address records nested inside the contact. Then real data shows up:

- Alice and her husband share a home address. With nesting it's stored twice, and fixing a typo in one copy leaves
  the other wrong.
- "Who lives at 10 Downing Street?" means scanning every contact's array.
- The label belongs to neither side. It isn't a property of the address (the same flat is "home" for Alice and
  "office" for her lodger), and it isn't a property of the contact.

The framework has **no arrays and no nested records**. Every collection is a **relation**: a set of *entries*, each
linking objects together and optionally carrying its own properties (like the label). Anything an array, a `Map` or
a reference does, a relation does too. Unlike them, a relation is visible from every object it links.

In [1]:
import { Proxies, Schemas } from "../src/Framework/index.js";
import { entries } from "./toolkit.js";

## Step 1: the objects

First, the things that exist on their own: contacts and addresses. There's nothing new here yet.

In [2]:
let Contact = new Schemas.OfObject.Builder()
  .properties(
    (p) => p.name("given_name").of((t) => t.as_native(String)),
    (p) => p.name("family_name").of((t) => t.as_native(String)),
  )
  .create();

let Address = new Schemas.OfObject.Builder()
  .properties(
    (p) => p.name("street").of((t) => t.as_native(String)),
    (p) => p.name("locality").of((t) => t.as_native(String)),
    (p) => p.name("postal_code").of((t) => t.as_native(String)),
    (p) => p.name("country_code").of((t) => t.as_native(String)), // ISO 3166-1 alpha-2
  )
  .create();

## Step 2: the relation

A relation names its **links** (one per object an entry connects) and its own **properties**. Notice that the links
have no types. `ContactAddresses` doesn't say *what* fills `contact` or `address`. It only says an entry has two
ends with those names, plus a `label`.

In [3]:
const ContactAddresses = new Schemas.OfRelation.Builder()
  .links("contact", "address")
  .properties((p) => p.name("label").of((t) => t.as_native(String))) // 'home', 'work', ...
  .create();

## Step 3: each object declares its side

The objects say which relations they take part in, and through which link. That's an *adjacency*:

- `name` is the accessor you'll use on the object (`.addresses(...)`),
- `of` is the relation,
- `me` is the link this object fills.

The schemas already exist, so they are edited in place with a builder started from them, finished with `update()`.
You saw the same `update()` on instances in case study 1.

In [4]:
Contact = new Schemas.OfObject.Builder(Contact)
  .relations((adj) => adj.name("addresses").of(ContactAddresses).me("contact"))
  .update();
Address = new Schemas.OfObject.Builder(Address)
  .relations((adj) => adj.name("residents").of(ContactAddresses).me("address"))
  .update();
console.log(Contact.validate(), Address.validate(), ContactAddresses.validate());

[] [] []


### Why three steps?

The contact schema mentions the relation, and the relation's meaning depends on the objects on each side. Something
has to exist first. Keeping links untyped and letting each object declare its own adjacencies breaks the cycle:

1. objects, 2. relations, 3. adjacencies added with `new Builder(existing)...update()`.

This is also why building never validates: halfway through step 3 the schemas are legitimately incomplete.

The same untyped link can be filled by more than one kind of object. A company could declare `.me("contact")` on
`ContactAddresses` too. Case study 4 comes back to what that means.

## Registering

Relations are registered too, because serialized data will name them. There is no builder for a relation, though.
You'll see why in a moment.

In [5]:
Proxies.register("Contact", Contact);
Proxies.register("Address", Address);
Proxies.register("ContactAddresses", ContactAddresses);
const Builders = Proxies.Builders;

try {
  Builders.ContactAddresses();
} catch (error) {
  console.log(String(error));
}

TypeError: 'ContactAddresses' is a relation; no relation builder is exposed


## Adding entries

Entries are added *from an object*, through its adjacency accessor. The accessor takes an **entry Spec**: a function
that receives an *entry builder* `x`. The object fills its own link (`contact`) by itself. You set the other links
and the entry's properties, fluently, with the same one-setter-per-name style as instances:

In [6]:
const downing = Builders.Address()
  .street("10 Downing Street").locality("London").postal_code("SW1A 2AA").country_code("GB")
  .create();

const alice = Builders.Contact()
  .given_name("Alice")
  .family_name("Liddell")
  .addresses((x: any) => x.address(downing).label("home"))
  .create();

const [row] = entries(alice, "addresses");
console.log(row?.address.street, "|", row?.label);

10 Downing Street | home


`entries` is a small helper, explained next. It returns one plain object per entry, mapping each link to the linked
object and each property to its value.

This is why relations have no builder of their own: an entry always belongs to the objects it links, so it's
created from one of them, which fills its own end.

### An aside: how `entries` reads data

Instances expose properties as fields, but their entries aren't fields: an entry is shared by several objects, so it
doesn't belong to any one of them. You read entries the same way *any* schema-agnostic tool reads data, by
**visiting**. A builder started from an instance is a visitor over that instance's data. Its methods don't return
values. They pass *handles* to callbacks:

```ts
Builders.Contact(alice).adjacency("addresses", (a: OfAdjacency) =>   // a: the adjacency
  a.entries((e) =>                                                   // e: one entry
    e.links((k) => k.target(console.log))));                         // k: one link; target: the object
```

Callbacks instead of return values let the same protocol drive readers, writers, serializers and validators without
building intermediate arrays. These handles are fully typed (`Visitors.OfAdjacency`, `OfEntry`, `OfLink`), so only
the first one needs its type written. The helper in `toolkit.ts` wraps this into an array of rows:

```ts
function read(entry: OfEntry): Row {
  const row: Row = {};
  entry.links((link) => link.target((target) => (row[link.name()] = target)));
  entry.properties((prop) => prop.value((v) => v.as_native((n) => (row[prop.name()] = n.get()))));
  return row;
}

export function entries(obj: Proxies.Instance, adjacency: string): Row[] {
  const rows: Row[] = [];
  Proxies.Builders[obj.schema_name()](obj).adjacency(adjacency, (a: OfAdjacency) => a.entries((e) => rows.push(read(e))));
  return rows;
}
```

## Shared addresses: one entry, two views

Now the husband, Reginald, shares the same address object. With nested arrays this would be a copy. Here it's
another entry pointing at the same address:

In [7]:
const reggie = Builders.Contact()
  .given_name("Reginald")
  .family_name("Hargreaves")
  .addresses((x: any) => x.address(downing).label("home"))
  .create();

console.log("residents:", entries(downing, "residents").map((e) => [e.contact.given_name, e.label]));

residents: [ [ "Alice", "home" ], [ "Reginald", "home" ] ]


"Who lives here?" needed no scan and no second index. Each entry is **one fact** seen from both of its ends: Alice's
`addresses` and the address's `residents` are two views of the same entries.

## Creating the other end inline

A link takes an existing object *or* a Spec that builds a new one on the spot:

In [8]:
Builders.Contact(alice)
  .addresses((x: any) =>
    x.address((a: any) => a.street("1 Infinite Loop").locality("Cupertino").country_code("US")).label("work"))
  .update();

console.log(entries(alice, "addresses").map((e) => [e.address.street, e.label]));

[ [ "10 Downing Street", "home" ], [ "1 Infinite Loop", "work" ] ]


How did the framework know the new object is an `Address`? The link is untyped, but exactly one registered schema
declares an adjacency filling `address` in `ContactAddresses`. That inference must be unambiguous. If two schemas
filled the link, inline creation would be refused and you'd pass an existing object instead (case study 4 shows
this).

## Entries form a set

Adding an entry equal to an existing one does nothing. Equal means the same linked objects (by identity) and equal
property values. An entry with a different label is a different fact, and so is one with no label at all:

In [9]:
Builders.Contact(reggie).addresses((x: any) => x.address(downing).label("home")).update(); // same fact: elided
Builders.Contact(reggie).addresses((x: any) => x.address(downing).label("office")).update(); // a different fact
Builders.Contact(reggie).addresses((x: any) => x.address(downing)).update(); // no label: different again
console.log(entries(reggie, "addresses").map((e) => e.label ?? "(no label)").sort());

[ "(no label)", "home", "office" ]


**Why a set?** An array could hold the same fact twice, and then removing "it" or counting "them" becomes
ambiguous. A set can't. If you really want two otherwise identical entries, the relation is missing a property that
tells them apart, such as a `since` date or a sequence number.

## Phones: why a one-link relation is not allowed

Phone numbers are the next thing to model, and the tempting shortcut is a relation with a single link whose entries
hold the number. The framework rejects that. An entry that carries data but links only one object is really an
object pretending to be a relation. Model the phone number as an object and link it:

In [10]:
const Shortcut = new Schemas.OfRelation.Builder()
  .links("contact")
  .properties((p) => p.name("number").of((t) => t.as_native(String)))
  .create();
console.log(Shortcut.validate());

[
  "a relation needs at least two links; a one-link relation merges a relation and an object"
]


In [11]:
let PhoneNumber = new Schemas.OfObject.Builder().properties((p) => p.name("number").of((t) => t.as_native(String))).create();
const ContactPhones = new Schemas.OfRelation.Builder()
  .links("contact", "phone")
  .properties((p) => p.name("label").of((t) => t.as_native(String)))
  .create();
PhoneNumber = new Schemas.OfObject.Builder(PhoneNumber).relations((adj) => adj.name("contacts").of(ContactPhones).me("phone")).update();
Contact = new Schemas.OfObject.Builder(Contact).relations((adj) => adj.name("phones").of(ContactPhones).me("contact")).update();
Proxies.register("PhoneNumber", PhoneNumber);
Proxies.register("ContactPhones", ContactPhones);

Builders.Contact(alice).phones((x: any) => x.phone((p: any) => p.number("+447700900123")).label("mobile")).update();
console.log(entries(alice, "phones").map((e) => [e.phone.number, e.label]));

[ [ "+447700900123", "mobile" ] ]


Two things happened there. Adding `phones` to `Contact` with `update()` changed the registered schema *in place*,
so existing contacts gained the new adjacency. And the switchboard number a whole office shares is now one object
linked from many contacts, just like the shared address.

## The rules you met

Mistakes in entries are caught while you build them:

In [12]:
for (const [attempt, why] of [
  [() => Builders.Contact().addresses((x: any) => x.contact(alice)), "the object's own link is implied"],
  [() => Builders.Contact().addresses((x: any) => x.address(downing).since("2020")), "not a property of the relation"],
  [() => Builders.Contact().addresses((x: any) => x.label("home")).create(), "the 'address' link is never set"],
] as const) {
  try {
    attempt();
  } catch (error) {
    console.log(`${why.padEnd(35)} -> ${String(error)}`);
  }
}

the object's own link is implied    -> AttributeError: 'contact' is not a link this entry can set


not a property of the relation      -> AttributeError: since


the 'address' link is never set     -> ValueError: link 'address' is not set


## What you learned

- There are no arrays or nested records. Collections are **relations** whose entries link objects and carry
  properties.
- Links are untyped names. Objects declare adjacencies with `.relations((adj) => adj.name(..).of(..).me(..))`.
  Schemas are defined in phases: objects, relations, adjacencies.
- Entries are added from an object: `.addresses((x: any) => x.address(obj).label("home"))`. A link takes an object
  or a Spec that builds one.
- An entry is one fact visible from every end. Entries form a set, and duplicates are elided.
- Data is read by visiting. `toolkit.entries` is a ten-line visitor.

**Next:** case study 3 edits relations (removing entries, moving house, cloning a contact) and looks at what
`update()` really replaces.